# SupremeAI Tiny Brain — Kaggle Auto-Tuning (QLoRA)
Distills Top AI routing & human-behavior decisions into `Qwen/Qwen2.5-0.5B-Instruct` on Kaggle's free T4 GPU and pushes the LoRA adapter to Hugging Face Hub (`njelitltd/supremeai-tiny-brain-lora`).

In [ ]:
!pip install -q transformers datasets peft trl bitsandbytes accelerate huggingface_hub

In [ ]:
import os, torch
from datasets import load_dataset
from huggingface_hub import login
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from trl import SFTTrainer

HF_TOKEN = os.getenv("HF_API_KEY", "")
if HF_TOKEN:
    login(token=HF_TOKEN)

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_REPO = "njelitltd/supremeai-tiny-brain-lora"
DATASET_PATH = "tiny_brain_seed.jsonl"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map="auto")

dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_config,
    args=TrainingArguments(
        output_dir="./tiny-brain-out",
        per_device_train_batch_size=2,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=5,
        save_strategy="no",
    ),
)

trainer.train()
if HF_TOKEN:
    trainer.model.push_to_hub(OUTPUT_REPO)
    tokenizer.push_to_hub(OUTPUT_REPO)
    print("Pushed Tiny Brain LoRA to:", OUTPUT_REPO)
